# Assignment 3: Milestone I Natural Language Processing
## Task 2 and Task 3
#### Student Name:

1.   Vo Ngoc Dung - S4124370
2.   Tang Hoang Ha - S4147768
3.   Nguyen Anh Duc - S4136756
4.   Nguyen Quoc Trong Nghia - S3343711

Environment: Python 3 and Jupyter notebook

Libraries used:
- **pandas, numpy** — data manipulation and numerical operations
- **scikit-learn** — TfidfVectorizer, LinearRegression, RandomForestClassifier, cross-validation, evaluation metrics
- **gensim** — pretrained FastText word embeddings (`fasttext-wiki-news-subwords-300`)
- **collections.Counter** — efficient token frequency counting

## Introduction

This notebook implements two tasks:

**Task 2 — Feature Representation:** Convert the preprocessed reviews (from Task 1) into three numeric representations suitable for machine learning:
1. Sparse count vectors (bag-of-words using `vocab.txt`)
2. Unweighted average FastText word vectors (300-d)
3. TF-IDF weighted average FastText word vectors (300-d)

**Task 3 — Classification & Regression:** Train and evaluate models to answer two questions:
- **Q1 (Regression):** Can review text predict `review_rating`? Evaluated with Linear Regression on FastText features.
- **Q2 (Classification):** Can we predict `is_a_buyer` from text + structured features? Evaluated with Random Forest using 5-fold stratified cross-validation.

All experiments use **5-fold cross-validation** to ensure robust, unbiased performance estimates.

## Importing Libraries

In [1]:
# !pip install -q --upgrade pip setuptools wheel
%pip install -q -r requirements.txt


[notice] A new release of pip is available: 23.1.2 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from collections import Counter
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, mean_absolute_error, mean_squared_error, r2_score
import os
import gensim.downloader as gensim_api
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## Task 2. Generating Feature Representations for Cosmetics/Beauty Reviews

**Purpose:** Machine learning models require numeric input. Raw text must be converted into fixed-length numeric vectors that capture the semantic content of each review. We generate three complementary representations:

| Representation | Type | Dimensionality | Captures |
|---------------|------|---------------|----------|
| Count vectors | Sparse | 5,701 (vocab size) | Exact word presence and frequency |
| Unweighted FastText | Dense | 300 | Semantic meaning via pretrained embeddings |
| Weighted FastText | Dense | 300 | Semantic meaning with TF-IDF importance weighting |

**Why multiple representations?**
- Count vectors are simple and interpretable but lose word order and semantic similarity (e.g., "moisturising" and "hydrating" are unrelated in count space).
- Word embeddings capture semantic relationships but compress all information into a fixed 300-d vector, potentially losing fine-grained term frequency information.
- Comparing both in Task 3 reveals which type of information (exact frequency vs. semantic meaning) is more useful for classification.

### 2.1 Count Vector Representation (Bag-of-Words)

**Purpose:** Encode each review as a sparse vector where each dimension corresponds to a word in `vocab.txt` and the value is the raw term frequency (number of times that word appears in the review).

**Why count vectors (not TF-IDF vectors)?**
- The assignment specifies raw count vectors as one of the three required outputs.
- Count vectors preserve the actual frequency signal — useful for models that can learn their own weighting internally (e.g., tree-based models).
- TF-IDF weighting is applied separately in the weighted embedding representation (Section 2.2), so we avoid double-weighting.

**Why sparse format?**
- With a vocabulary of 5,701 words and an average of ~7 tokens per review, most entries are zero. Storing only non-zero values reduces file size from ~350 MB (dense) to ~5 MB (sparse) — a ~70× compression.

**Implementation details:**
- Vocabulary loaded from `vocab.txt` (word → integer index, alphabetically sorted, produced by Task 1).
- Each review's preprocessed `review_text` (space-separated tokens) is tokenised and counted using `collections.Counter`.
- Only tokens present in the vocabulary are counted (out-of-vocabulary tokens are ignored).
- Output format: `#review_index,word_index:frequency,...` (sorted by word index for consistency).
- Saved to `count_vectors.txt`.

In [3]:
PROCESSED_CSV = "processed.csv"
VOCAB = "vocab.txt"
COUNT_VECTOR = "count_vectors.txt"


def GenerateCountVector():
    # Load vocabulary (word -> integer index)
    vocab = {}
    with open(VOCAB, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            word, idx = line.rsplit(":", 1)
            vocab[word] = int(idx)

    # Load processed reviews
    df = pd.read_csv(PROCESSED_CSV)
    # review_text contains space-separated tokens produced by Task 1
    review_texts = df["review_text"].fillna("").astype(str).tolist()

    # Build and write sparse count vectors
    with open(COUNT_VECTOR, "w", encoding="utf-8") as out:
        for review_idx, text in enumerate(review_texts):
            tokens = text.split()
            # Count only tokens that exist in the vocabulary
            counts = Counter(token for token in tokens if token in vocab)
            # Sort by word integer index for a consistent ordering
            sparse_entries = sorted(
                (vocab[word], freq) for word, freq in counts.items()
            )
            sparse_str = ",".join(f"{idx}:{freq}" for idx, freq in sparse_entries)
            out.write(f"#{review_idx},{sparse_str}\n")

    print(f"Count vectors saved to '{COUNT_VECTOR}' ({len(review_texts)} reviews).")


# Run
GenerateCountVector()

Count vectors saved to 'count_vectors.txt' (61284 reviews).


**Observation:** All 61,284 reviews were successfully converted to sparse count vectors. The sparse format ensures efficient storage and fast loading for Task 3 experiments. Each line contains only the non-zero word indices and their frequencies, which aligns with the short average review length (~7 tokens) identified in Task 1.

### 2.2 Word Embedding Vectors (FastText — Unweighted & Weighted)

**Purpose:** Represent each review as a dense 300-dimensional vector by averaging pretrained word embeddings, producing a fixed-length numeric representation that captures semantic meaning.

**Why FastText instead of Word2Vec or GloVe?**

| Model | Advantage | Limitation |
|-------|-----------|-----------|
| **FastText** (chosen) | Handles **subword information** — can generate vectors for misspellings or rare morphological variants (e.g., "moisturis") by composing character n-grams. Critical for our stemmed/lemmatised tokens. | Larger model file (~1 GB). |
| Word2Vec | Fast, well-established. | No subword handling — out-of-vocabulary words get zero vectors. |
| GloVe | Good for capturing global co-occurrence patterns. | No subword handling; less suited for noisy user-generated text. |

**Why `fasttext-wiki-news-subwords-300`?**
- Trained on Wikipedia + news corpus — broad vocabulary coverage.
- 300 dimensions — standard size balancing expressiveness and computational cost.
- Subword capability means our stemmed tokens (e.g., "moisturis", "cleanser") are still representable even if not in the model's exact vocabulary.

**Two averaging strategies:**
1. **Unweighted (simple mean):** Each token contributes equally. Simple baseline that works well when reviews are short (ours average ~7 tokens).
2. **TF-IDF weighted mean:** Each token's vector is scaled by its corpus-wide TF-IDF weight before averaging. This gives more influence to informative/rare terms and less to common ones.

**Why average pooling instead of alternatives?**

| Alternative | Why not used |
|-------------|-------------|
| Concatenation | Would produce variable-length vectors (reviews have different lengths). |
| Max pooling | Loses frequency information; less interpretable. |
| Document embeddings (Doc2Vec) | Requires training on our corpus; pretrained models not readily available. |
| Sentence transformers (BERT) | Computationally expensive; out of scope for this assignment. |

**Output format:** `#review_index,v0,v1,...,v299` — one line per review, 300 comma-separated float values.

In [4]:
UNWEIGHTED_VECTOR = "unweighted_vectors.txt"
WEIGHTED_VECTOR = "weighted_vectors.txt"
FASTTEXT_MODEL_NAME = "fasttext-wiki-news-subwords-300"


def GenerateEmbeddingVectors():
    # Skip regeneration when both output files already exist.
    if os.path.exists(UNWEIGHTED_VECTOR) and os.path.exists(WEIGHTED_VECTOR):
        print(f"Skip GenerateEmbeddingVectors: '{UNWEIGHTED_VECTOR}' and '{WEIGHTED_VECTOR}' already exist.")
        return

    # Load pretrained FastText model
    print(f"Loading FastText model '{FASTTEXT_MODEL_NAME}'")
    fasttext_model = gensim_api.load(FASTTEXT_MODEL_NAME)
    vector_size = fasttext_model.vector_size
    print(f"Model loaded. Vector size: {vector_size}")

    # Load processed reviews
    df = pd.read_csv(PROCESSED_CSV)
    review_texts = df["review_text"].fillna("").astype(str).tolist()
    print("Load review")
    # Fit TF-IDF over the full corpus (for weighted representation)
    # tokenizer=str.split preserves the already-cleaned tokens from Task 1
    tfidf = TfidfVectorizer(tokenizer=str.split, lowercase=False, token_pattern=None)
    tfidf_matrix = tfidf.fit_transform(review_texts)
    tfidf_feature_names = tfidf.get_feature_names_out()
    tfidf_vocab = {word: idx for idx, word in enumerate(tfidf_feature_names)}

    # Generate and write vectors
    with open(UNWEIGHTED_VECTOR, "w", encoding="utf-8") as uw_out, open(
        WEIGHTED_VECTOR, "w", encoding="utf-8"
    ) as w_out:

        for review_idx, text in enumerate(review_texts):
            tokens = text.split()
            # Keep only tokens the FastText model knows
            valid_tokens = [t for t in tokens if t in fasttext_model]

            if valid_tokens:
                vectors = np.array([fasttext_model[t] for t in valid_tokens])

                # Unweighted: simple average of word vectors
                unweighted_vec = vectors.mean(axis=0)

                # Weighted: TF-IDF weighted average
                tfidf_row = tfidf_matrix[review_idx]
                weights = np.array(
                    [
                        tfidf_row[0, tfidf_vocab[t]] if t in tfidf_vocab else 0.0
                        for t in valid_tokens
                    ]
                )
                weight_sum = weights.sum()
                if weight_sum > 0:
                    weighted_vec = (vectors * weights[:, np.newaxis]).sum(
                        axis=0
                    ) / weight_sum
                else:
                    weighted_vec = unweighted_vec
            else:
                unweighted_vec = np.zeros(vector_size)
                weighted_vec = np.zeros(vector_size)

            uw_out.write(
                f"#{review_idx}," + ",".join(f"{v:.6f}" for v in unweighted_vec) + "\n"
            )
            w_out.write(
                f"#{review_idx}," + ",".join(f"{v:.6f}" for v in weighted_vec) + "\n"
            )

    print(f"Number of reviews: {len(review_texts)}")
    print(f"Unweighted vectors saved to '{UNWEIGHTED_VECTOR}'")
    print(f"Weighted vectors saved to '{WEIGHTED_VECTOR}'")


# Run
GenerateEmbeddingVectors()

Loading FastText model 'fasttext-wiki-news-subwords-300'
[==================================================] 100.0% 958.5/958.4MB downloaded
Model loaded. Vector size: 300
Load review
Number of reviews: 61284
Unweighted vectors saved to 'unweighted_vectors.txt'
Weighted vectors saved to 'weighted_vectors.txt'


**Observation:** Both unweighted and weighted vectors were generated for all 61,284 reviews. The FastText subword capability ensured that stemmed tokens (produced by Task 1's Porter Stemmer) still received meaningful embeddings rather than zero vectors. The TF-IDF weighting was fitted across the full corpus to capture global term importance, ensuring that common but uninformative words contribute less to the document representation.

## Task 3. Question 1: Predicting Review Rating (Regression)

**Question:** Can the semantic content of a review (captured by FastText embeddings) predict the numeric star rating (`review_rating`, 1–5) that the reviewer assigned?

**Why Linear Regression?**
- Serves as a simple, interpretable **baseline** — if a linear model can explain some variance, the features contain genuine signal.
- Fast to train and evaluate, enabling quick iteration.
- The target (`review_rating`) is numeric and approximately continuous, making regression appropriate.

**Why FastText embeddings (not count vectors)?**
- Count vectors are 5,701-dimensional and sparse — Linear Regression with many sparse features is prone to overfitting without regularisation.
- Dense 300-d embeddings provide a compact, regularised representation that linear models can efficiently learn from.

**Evaluation setup:**
- **5-fold cross-validation** (KFold, shuffle=True) — ensures every review is used for both training and testing exactly once.
- **Metrics:** MAE (average magnitude of error), MSE/RMSE (penalises large errors), R² (proportion of variance explained).
- Both unweighted and weighted FastText vectors are compared to assess whether TF-IDF weighting improves rating prediction.

**Context from EDA (Task 1):** The correlation analysis showed `review_rating` has negligible correlation with `is_a_buyer` (r = 0.029), but here we ask the reverse question — whether the *text content* can predict the rating itself.

In [5]:
def load_dense_vectors(file_path: str) -> np.ndarray:
    vectors = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(",")
            vec = [float(v) for v in parts[1:]]  # Skip #review_index
            vectors.append(vec)
    return np.array(vectors, dtype=np.float32)


# Load target
df = pd.read_csv(PROCESSED_CSV)
y_all = pd.to_numeric(df["review_rating"], errors="coerce")
valid_mask = y_all.notna().to_numpy()
y = y_all[valid_mask].to_numpy(dtype=np.float32)

# Load generated feature vectors
X_unweighted = load_dense_vectors(UNWEIGHTED_VECTOR)[valid_mask]
X_weighted = load_dense_vectors(WEIGHTED_VECTOR)[valid_mask]


def evaluate_linear_regression_5fold(X: np.ndarray, y: np.ndarray, name: str):
    """Evaluate Linear Regression using 5-fold cross-validation."""
    kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    print(f"\n{name} (5-Fold Cross-Validation Results)")
    mae_scores = []
    mse_scores = []
    rmse_scores = []
    r2_scores = []

    fold_num = 1
    for train_idx, test_idx in kf.split(X):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        model = LinearRegression()
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        mae = mean_absolute_error(y_test, y_pred)
        mse = mean_squared_error(y_test, y_pred)
        rmse = float(np.sqrt(mse))
        r2 = r2_score(y_test, y_pred)

        mae_scores.append(mae)
        mse_scores.append(mse)
        rmse_scores.append(rmse)
        r2_scores.append(r2)

        print(
            f"  Fold {fold_num}: MAE={mae:.4f}, MSE={mse:.4f}, RMSE={rmse:.4f}, R²={r2:.4f}"
        )
        fold_num += 1

    print(f"MAE  : {np.mean(mae_scores):.4f} ± {np.std(mae_scores):.4f}")
    print(f"MSE  : {np.mean(mse_scores):.4f} ± {np.std(mse_scores):.4f}")
    print(f"RMSE : {np.mean(rmse_scores):.4f} ± {np.std(rmse_scores):.4f}")
    print(f"R²   : {np.mean(r2_scores):.4f} ± {np.std(r2_scores):.4f}")


print(f"Samples used: {len(y)}")
print(f"Feature size (unweighted): {X_unweighted.shape[1]}")
print(f"Feature size (weighted)  : {X_weighted.shape[1]}")

evaluate_linear_regression_5fold(
    X_unweighted, y, "Linear Regression with Unweighted FastText"
)
evaluate_linear_regression_5fold(
    X_weighted, y, "Linear Regression with Weighted FastText"
)

Samples used: 61283
Feature size (unweighted): 300
Feature size (weighted)  : 300

Linear Regression with Unweighted FastText (5-Fold Cross-Validation Results)
  Fold 1: MAE=0.6887, MSE=0.9034, RMSE=0.9504, R²=0.1924
  Fold 2: MAE=0.6928, MSE=0.9071, RMSE=0.9524, R²=0.1937
  Fold 3: MAE=0.6886, MSE=0.8907, RMSE=0.9438, R²=0.2038
  Fold 4: MAE=0.6915, MSE=0.9040, RMSE=0.9508, R²=0.2060
  Fold 5: MAE=0.6951, MSE=0.9140, RMSE=0.9560, R²=0.2010
MAE  : 0.6913 ± 0.0025
MSE  : 0.9038 ± 0.0076
RMSE : 0.9507 ± 0.0040
R²   : 0.1994 ± 0.0054

Linear Regression with Weighted FastText (5-Fold Cross-Validation Results)
  Fold 1: MAE=0.6923, MSE=0.9127, RMSE=0.9554, R²=0.1840
  Fold 2: MAE=0.6975, MSE=0.9185, RMSE=0.9584, R²=0.1836
  Fold 3: MAE=0.6934, MSE=0.9027, RMSE=0.9501, R²=0.1931
  Fold 4: MAE=0.6964, MSE=0.9175, RMSE=0.9578, R²=0.1942
  Fold 5: MAE=0.6992, MSE=0.9254, RMSE=0.9620, R²=0.1910
MAE  : 0.6958 ± 0.0026
MSE  : 0.9154 ± 0.0075
RMSE : 0.9567 ± 0.0039
R²   : 0.1892 ± 0.0045


### Findings: Linear Regression for Review Rating Prediction

#### Results Comparison

| Feature Set | MAE | RMSE | R² |
|------------|-----|------|-----|
| Unweighted FastText | 0.6913 ± 0.0025 | 0.9507 ± 0.0040 | 0.1994 ± 0.0054 |
| Weighted FastText | 0.6958 ± 0.0026 | 0.9567 ± 0.0039 | 0.1892 ± 0.0045 |

#### Interpretation

1. **Modest predictive power:** R² ≈ 0.19–0.20 means the text embeddings explain roughly **20% of the variance** in review ratings. This is non-trivial but indicates that star ratings are only partially recoverable from text content alone.

2. **Unweighted slightly outperforms weighted:** The simple average embedding (R² = 0.1994) marginally beats TF-IDF weighted (R² = 0.1892). This suggests that for short reviews (~7 tokens), equal weighting is sufficient — TF-IDF weighting may introduce noise when there are too few tokens to reliably estimate term importance.

3. **MAE ≈ 0.69:** On a 1–5 scale, the model's predictions are off by less than 1 star on average. Given that the rating distribution is heavily skewed toward 5 stars (as observed in Task 1 EDA), this is a reasonable baseline.

4. **Why only ~20% R²?** Star ratings are inherently subjective — two reviewers may write similar text but assign different stars. Also, much rating signal may come from *sentiment intensity* and *aspect-specific opinions* that simple averaging of embeddings cannot fully capture.

#### Conclusion
Text embeddings contain meaningful signal for rating prediction, but a linear model on averaged embeddings has clear limitations. More complex architectures (e.g., attention-based models, or using the full sequence of word vectors) would likely improve performance, but are outside the scope of this task.

## Task 3. Question 2: Predicting Buyer Status (Classification)

**Question:** Can we predict whether a reviewer is a verified buyer (`is_a_buyer`) using text features combined with structured product metadata?

**Why Random Forest?**
- Handles **mixed feature types** naturally — can process both dense embeddings (300 continuous features) and structured features (categorical/numeric) without separate preprocessing.
- Non-linear model — can capture interactions between features that Linear Regression would miss (e.g., "low price + positive text = likely buyer").
- Robust to overfitting with many features due to ensemble averaging and feature subsampling.
- Provides built-in feature importance rankings for interpretability.

**Why not Logistic Regression or SVM?**
- Logistic Regression assumes linear decision boundaries — our EDA showed weak linear correlations, suggesting non-linear patterns may be needed.
- SVM with non-linear kernels would be computationally expensive on 61,284 samples × 304 features.

**Feature configuration (informed by EDA):**
- **Embeddings:** 300-d FastText vectors (unweighted and weighted, compared separately).
- **Structured features (4):** `price`, `product_rating_count`, `brand_encoded`, `product_encoded` — selected based on Task 1 EDA correlation analysis as the features with strongest signal.
- **Excluded:** `review_rating` and `avg_product_rating` — EDA showed negligible correlation with `is_a_buyer` (r = 0.029 and 0.044 respectively).
- **Excluded:** `brand_encoded` + `product_encoded` simultaneously could introduce multicollinearity (r = 0.934 from EDA), but Random Forest is robust to this, so both are retained.

**Hyperparameters:**
- `n_estimators=300` — sufficient ensemble size for stable predictions.
- `class_weight="balanced"` — addresses the ~3.7:1 class imbalance by upweighting the minority class (non-buyers).
- `n_jobs=-1` — parallelises across all CPU cores for faster training.

**Evaluation:** 5-fold **Stratified** cross-validation (preserves class ratio in each fold) with Accuracy, Precision, Recall, and F1-score.

In [6]:
USE_STRUCTURED_FEATURES = True
STRUCTURED_FEATURES = [
    # "review_rating",
    "price",
    # "avg_product_rating",
    "product_rating_count",
    "brand_encoded",
    "product_encoded",
]

RF_PARAMS = {
    "n_estimators": 300,
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "class_weight": "balanced",
}


def load_dense_vectors(file_path: str) -> np.ndarray:
    vectors = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(",")
            vectors.append([float(v) for v in parts[1:]])  # skip #review_index
    return np.array(vectors, dtype=np.float32)


def build_structured_matrix(
    df_in: pd.DataFrame, feature_names: list[str]
) -> np.ndarray:
    cols = []

    for feat in feature_names:
        if feat == "brand_encoded":
            encoded = pd.factorize(df_in["brand_name"].fillna("unknown"))[0].astype(
                np.float32
            )
            cols.append(encoded.reshape(-1, 1))
        elif feat == "product_encoded":
            encoded = pd.factorize(df_in["product_title"].fillna("unknown"))[0].astype(
                np.float32
            )
            cols.append(encoded.reshape(-1, 1))
        else:
            if feat not in df_in.columns:
                raise ValueError(f"Feature '{feat}' not found in dataframe columns.")
            numeric_col = pd.to_numeric(df_in[feat], errors="coerce")
            filled_col = numeric_col.fillna(numeric_col.median()).to_numpy(
                dtype=np.float32
            )
            cols.append(filled_col.reshape(-1, 1))

    return np.hstack(cols) if cols else np.empty((len(df_in), 0), dtype=np.float32)


def prepare_binary_target(series: pd.Series) -> np.ndarray:
    if series.dtype == bool:
        return series.astype(int).to_numpy()

    numeric_try = pd.to_numeric(series, errors="coerce")
    if numeric_try.notna().all():
        return numeric_try.astype(int).to_numpy()

    str_series = series.astype(str).str.strip().str.lower()
    mapping = {"true": 1, "false": 0, "yes": 1, "no": 0, "1": 1, "0": 0}
    mapped = str_series.map(mapping)

    if mapped.isna().any():
        bad_vals = sorted(str_series[mapped.isna()].unique().tolist())[:5]
        raise ValueError(
            f"Unable to map some label values in is_a_buyer, examples: {bad_vals}"
        )

    return mapped.astype(int).to_numpy()


def evaluate_rf_variant_5fold(
    name: str, X_embed: np.ndarray, y: np.ndarray, X_struct: np.ndarray | None
):
    print(f"\n{name} (5-Fold Stratified Cross-Validation Results)")
    """Evaluate Random Forest using 5-fold stratified cross-validation."""
    if X_struct is not None and X_struct.shape[1] > 0:
        X_all = np.hstack([X_embed, X_struct])
    else:
        X_all = X_embed

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    acc_scores = []
    prec_scores = []
    rec_scores = []
    f1_scores = []

    fold_num = 1
    for train_idx, test_idx in skf.split(X_all, y):
        X_train, X_test = X_all[train_idx], X_all[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        model = RandomForestClassifier(**RF_PARAMS)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)

        acc_scores.append(acc)
        prec_scores.append(prec)
        rec_scores.append(rec)
        f1_scores.append(f1)

        print(
            f"  Fold {fold_num}: Acc={acc:.4f}, Prec={prec:.4f}, Rec={rec:.4f}, F1={f1:.4f}"
        )
        fold_num += 1

    print(f"Samples: {len(y)} | Features: {X_all.shape[1]}")
    print(f"Accuracy : {np.mean(acc_scores):.4f} ± {np.std(acc_scores):.4f}")
    print(f"Precision: {np.mean(prec_scores):.4f} ± {np.std(prec_scores):.4f}")
    print(f"Recall   : {np.mean(rec_scores):.4f} ± {np.std(rec_scores):.4f}")
    print(f"F1-score : {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")


# Load data
df_model = pd.read_csv(PROCESSED_CSV)
y = prepare_binary_target(df_model["is_a_buyer"])

# Optional structured matrix
X_structured = None
if USE_STRUCTURED_FEATURES:
    X_structured = build_structured_matrix(df_model, STRUCTURED_FEATURES)
    print(f"Using structured features: {STRUCTURED_FEATURES}")
    print(f"Structured matrix shape: {X_structured.shape}")
else:
    print("Using embeddings only (structured features disabled).")

# Evaluate both embedding variants
X_unweighted = load_dense_vectors(UNWEIGHTED_VECTOR)
X_weighted = load_dense_vectors(WEIGHTED_VECTOR)

if len(y) != len(X_unweighted) or len(y) != len(X_weighted):
    raise ValueError("Row count mismatch among labels and embedding vectors.")

Using structured features: ['price', 'product_rating_count', 'brand_encoded', 'product_encoded']
Structured matrix shape: (61284, 4)


In [7]:
evaluate_rf_variant_5fold(
    "Random Forest with Unweighted FastText",
    X_unweighted,
    y,
    X_structured,
)

evaluate_rf_variant_5fold(
    "Random Forest with Weighted FastText",
    X_weighted,
    y,
    X_structured,
)


Random Forest with Unweighted FastText (5-Fold Stratified Cross-Validation Results)
  Fold 1: Acc=0.8014, Prec=0.8187, Rec=0.9603, F1=0.8839
  Fold 2: Acc=0.7959, Prec=0.8179, Rec=0.9526, F1=0.8802
  Fold 3: Acc=0.8042, Prec=0.8239, Rec=0.9553, F1=0.8848
  Fold 4: Acc=0.8023, Prec=0.8210, Rec=0.9576, F1=0.8840
  Fold 5: Acc=0.8000, Prec=0.8187, Rec=0.9580, F1=0.8829
Samples: 61284 | Features: 304
Accuracy : 0.8008 ± 0.0028
Precision: 0.8200 ± 0.0022
Recall   : 0.9568 ± 0.0026
F1-score : 0.8831 ± 0.0016

Random Forest with Weighted FastText (5-Fold Stratified Cross-Validation Results)
  Fold 1: Acc=0.8017, Prec=0.8206, Rec=0.9572, F1=0.8837
  Fold 2: Acc=0.7971, Prec=0.8189, Rec=0.9529, F1=0.8808
  Fold 3: Acc=0.8047, Prec=0.8234, Rec=0.9571, F1=0.8852
  Fold 4: Acc=0.8016, Prec=0.8208, Rec=0.9567, F1=0.8835
  Fold 5: Acc=0.7998, Prec=0.8201, Rec=0.9551, F1=0.8824
Samples: 61284 | Features: 304
Accuracy : 0.8010 ± 0.0025
Precision: 0.8208 ± 0.0015
Recall   : 0.9558 ± 0.0016
F1-score : 

### Findings: Random Forest Classification for Buyer Prediction

#### Results Comparison

| Feature Set | Accuracy | Precision | Recall | F1-score |
|------------|----------|-----------|--------|----------|
| Unweighted FastText + Structured | 0.8008 ± 0.0028 | 0.8200 ± 0.0022 | 0.9568 ± 0.0026 | 0.8831 ± 0.0016 |
| Weighted FastText + Structured | 0.8010 ± 0.0025 | 0.8208 ± 0.0015 | 0.9558 ± 0.0016 | 0.8831 ± 0.0014 |

#### Interpretation

1. **Strong classification performance:** F1 ≈ 0.883 indicates the model effectively distinguishes buyers from non-buyers. With 80% accuracy on a dataset with ~79% majority class, the model provides meaningful predictions beyond simply predicting the majority class (which would achieve ~79% accuracy but poor recall on the minority class).

2. **High recall (0.957) vs moderate precision (0.821):** The model correctly identifies 95.7% of actual buyers but also misclassifies some non-buyers as buyers (precision = 82%). This asymmetry reflects the `class_weight="balanced"` setting, which prioritises recall by penalising missed buyers more heavily.

3. **Unweighted ≈ Weighted:** Both embedding types produce virtually identical results (F1 = 0.8831 for both). This reinforces the finding from Q1 — for short reviews, simple averaging is as effective as TF-IDF weighting.

4. **Low variance across folds:** Standard deviations are all < 0.003, indicating stable, generalisable performance that is not sensitive to particular train/test splits.

5. **Connection to EDA findings:** The EDA in Task 1 showed that structured features alone have weak correlations with `is_a_buyer` (max |r| = 0.21). The strong F1 here confirms that **text features (embeddings) provide the dominant signal**, while structured features contribute incremental improvement through non-linear interactions captured by Random Forest.

#### Limitations
- We did not test embeddings-only vs structured-only to quantify each component's individual contribution.
- Random Forest does not natively handle sequential text — it treats the 300 embedding dimensions independently, potentially missing word-order patterns.
- The model's precision gap (82% vs 96% recall) means ~18% of predicted buyers are actually non-buyers, which may matter depending on the business application.

## Summary

### Task 2: Feature Representation Outputs

| Output File | Format | Description |
|------------|--------|-------------|
| `count_vectors.txt` | Sparse: `#idx,word_idx:freq,...` | 61,284 reviews × 5,701 vocab — raw term frequencies |
| `unweighted_vectors.txt` | Dense: `#idx,v0,...,v299` | 61,284 reviews × 300d — simple average of FastText embeddings |
| `weighted_vectors.txt` | Dense: `#idx,v0,...,v299` | 61,284 reviews × 300d — TF-IDF weighted average of FastText embeddings |

### Task 3: Key Findings

| Question | Model | Best Metric | Key Insight |
|----------|-------|-------------|-------------|
| Q1: Predict `review_rating` | Linear Regression | R² = 0.1994 (Unweighted) | Text explains ~20% of rating variance; unweighted embeddings slightly outperform weighted for short reviews. |
| Q2: Predict `is_a_buyer` | Random Forest | F1 = 0.8831 (both) | Strong classification with text + structured features; embeddings provide dominant signal, structured features add incremental value through non-linear interactions. |

### Design Decisions Justified by EDA (Task 1)

- **Feature selection for Q2:** Excluded `review_rating` and `avg_product_rating` (negligible r with target) — retained `price`, `product_rating_count`, `brand_encoded`, `product_encoded`.
- **Class imbalance handling:** Used `class_weight="balanced"` in Random Forest, informed by the ~3.7:1 imbalance observed in EDA.
- **Random split justified:** Temporal analysis showed stable buyer proportions across years — no time-based split needed.
- **Text features as primary signal:** EDA's weak structured-feature correlations predicted that embeddings would dominate classification power — confirmed by Task 3 results.

### Unweighted vs Weighted Embeddings

Across both tasks, **unweighted averaging performed equally or slightly better** than TF-IDF weighted averaging. This is likely because:
1. Reviews are short (~7 tokens on average), so there is little variance in term importance to exploit.
2. TF-IDF weights fitted on the full corpus may not precisely reflect importance within a single short review.

For longer documents, weighted embeddings would likely show a clearer advantage.